### Ekonometrický projekt — LS 2025/2026
**Analýza mzdovej štruktúry pracovníkov (Mincerov model)** | wage1 (Wooldridge, n=526)

In [1]:
# Nacitanie dat — happiness_europe.txt
df <- read.table("Data/happiness_europe.txt", sep="\t", header=TRUE,
                 check.names=FALSE, stringsAsFactors=FALSE)

names(df) <- c("Country", "Region", "Happiness_Rank", "Happiness_Score",
               "Std_Error", "Economy", "Family", "Health",
               "Freedom", "Trust", "Generosity", "Dystopia_Residual")

# Konverzia numerickych stlpcov
num_cols <- c("Happiness_Rank","Happiness_Score","Std_Error","Economy",
              "Family","Health","Freedom","Trust","Generosity","Dystopia_Residual")
df[num_cols] <- lapply(df[num_cols], as.numeric)

cat("Rozmer:", nrow(df), "x", ncol(df), "\n")
str(df)

Rozmer: 50 x 12 
'data.frame':	50 obs. of  12 variables:
 $ Country          : chr  "Switzerland" "Iceland" "Denmark" "Norway" ...
 $ Region           : chr  "Western Europe" "Western Europe" "Western Europe" "Western Europe" ...
 $ Happiness_Rank   : num  1 2 3 4 6 7 8 13 17 18 ...
 $ Happiness_Score  : num  7.59 7.56 7.53 7.52 7.41 ...
 $ Std_Error        : num  0.0341 0.0488 0.0333 0.0388 0.0314 ...
 $ Economy          : num  1.4 1.3 1.33 1.46 1.29 ...
 $ Family           : num  1.35 1.4 1.36 1.33 1.32 ...
 $ Health           : num  0.941 0.948 0.875 0.885 0.889 ...
 $ Freedom          : num  0.666 0.629 0.649 0.67 0.642 ...
 $ Trust            : num  0.42 0.141 0.484 0.365 0.414 ...
 $ Generosity       : num  0.297 0.436 0.341 0.347 0.234 ...
 $ Dystopia_Residual: num  2.52 2.7 2.49 2.47 2.62 ...


In [2]:
# MNS model: Happiness_Score ~ Family + Economy + Health + Freedom

Y <- df$Happiness_Score
X <- cbind(1, df$Family, df$Economy, df$Health, df$Freedom)
colnames(X) <- c("intercept", "Family", "Economy", "Health", "Freedom")

n <- nrow(X)
k <- ncol(X)

# odhad beta
betaHAT <- solve(t(X) %*% X) %*% t(X) %*% Y
rownames(betaHAT) <- colnames(X)
cat("Odhady beta:\n"); print(betaHAT)

# rezidua a odhad sigma^2
YHAT       <- X %*% betaHAT
epsHAT     <- Y - YHAT
s2         <- as.numeric(t(epsHAT) %*% epsHAT / (n - k))
cat("\nOdhad sigma^2 (s2):", s2, "\n")
cat("Odhad sigma  (s) :", sqrt(s2), "\n")

# R^2, RSS, ESS, TSS
RSS <- sum(epsHAT^2)
TSS <- sum((Y - mean(Y))^2)
ESS <- TSS - RSS
R2  <- ESS / TSS
cat("\nRSS:", RSS, " ESS:", ESS, " TSS:", TSS, "\n")
cat("R^2:", R2, "\n")

# overenie cez lm()
model <- lm(Happiness_Score ~ Family + Economy + Health + Freedom, data = df)
summary(model)

Odhady beta:
               [,1]
intercept 1.6833727
Family    1.1236625
Economy   0.9736115
Health    0.9508998
Freedom   2.5565038

Odhad sigma^2 (s2): 0.2416314 
Odhad sigma  (s) : 0.4915601 

RSS: 10.87341  ESS: 34.25526  TSS: 45.12868 
R^2: 0.7590576 



Call:
lm(formula = Happiness_Score ~ Family + Economy + Health + Freedom, 
    data = df)

Residuals:
    Min      1Q  Median      3Q     Max 
-1.2046 -0.1929 -0.0340  0.3589  1.0591 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept)   1.6834     0.6187   2.721  0.00922 ** 
Family        1.1237     0.4765   2.358  0.02278 *  
Economy       0.9736     0.5285   1.842  0.07204 .  
Health        0.9509     1.0309   0.922  0.36122    
Freedom       2.5565     0.5527   4.625 3.17e-05 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 0.4916 on 45 degrees of freedom
Multiple R-squared:  0.7591,	Adjusted R-squared:  0.7376 
F-statistic: 35.44 on 4 and 45 DF,  p-value: 2.24e-13


In [3]:
# Test kontrastu: H0: beta_Health = 2 * beta_Economy
# t.j. beta_Health - 2*beta_Economy = 0
# poradie: (intercept, Family, Economy, Health, Freedom)

a <- c(0, 0, -2, 1, 0)
r <- 0

X <- model.matrix(model)
T <- (t(a) %*% betaHAT - r) / sqrt(s2 * t(a) %*% solve(t(X)%*%X) %*% a)
krit <- qt(0.975, df = n-k)
p    <- 2 * (1 - pt(abs(T), df = n-k))

cat("T-statistika:    ", round(T, 4), "\n")
cat("Kritická hodnota:", round(krit, 4), "\n")
cat("p-hodnota:       ", round(p, 4), "\n\n")
if (p < 0.05) {
  cat("Záver: H0 ZAMIETAME (p < 0.05) — beta_Health != 2 * beta_Economy\n")
} else {
  cat("Záver: H0 NEZAMIETAME (p >= 0.05) — nedostatok dôkazov proti beta_Health = 2 * beta_Economy\n")
}

T-statistika:     -0.5105 
Kritická hodnota: 2.0141 
p-hodnota:        0.6122 

Záver: H0 NEZAMIETAME (p >= 0.05) — nedostatok dôkazov proti beta_Health = 2 * beta_Economy


In [4]:
# Test kontrastu: H0: beta_Family = beta_Economy + beta_Freedom
# t.j. beta_Family - beta_Economy - beta_Freedom = 0
# poradie: (intercept, Family, Economy, Health, Freedom)

a <- c(0, 1, -1, 0, -1)
r <- 0

X <- model.matrix(model)
T <- (t(a) %*% betaHAT - r) / sqrt(s2 * t(a) %*% solve(t(X)%*%X) %*% a)
krit <- qt(0.975, df = n-k)
p    <- 2 * (1 - pt(abs(T), df = n-k))

cat("T-statistika:    ", round(T, 4), "\n")
cat("Kritická hodnota:", round(krit, 4), "\n")
cat("p-hodnota:       ", round(p, 4), "\n\n")
if (p < 0.05) {
  cat("Záver: H0 ZAMIETAME (p < 0.05) — beta_Family != beta_Economy + beta_Freedom\n")
} else {
  cat("Záver: H0 NEZAMIETAME (p >= 0.05) — nedostatok dôkazov proti beta_Family = beta_Economy + beta_Freedom\n")
}

T-statistika:     -2.1511 
Kritická hodnota: 2.0141 
p-hodnota:        0.0369 

Záver: H0 ZAMIETAME (p < 0.05) — beta_Family != beta_Economy + beta_Freedom


In [ ]:
# Bonferroniho simultánne IS pre m=3 kontrasty
# poradie parametrov: (intercept, Family, Economy, Health, Freedom)
#
# Kontrasty (ekonomická interpretácia):
#   K1: beta_Freedom - beta_Family   → je sloboda dôležitejšia ako rodina?
#   K2: beta_Freedom - beta_Economy  → je sloboda dôležitejšia ako ekonomika?
#   K3: beta_Family  - beta_Economy  → je rodina dôležitejšia ako ekonomika?

alpha <- 0.05
m     <- 3

A <- list(
  K1 = c(0, -1,  0,  0,  1),   # Freedom - Family
  K2 = c(0,  0, -1,  0,  1),   # Freedom - Economy
  K3 = c(0,  1, -1,  0,  0)    # Family  - Economy
)

X      <- model.matrix(model)
covMAT <- s2 * solve(t(X) %*% X)
krit_B <- qt(1 - alpha / (2 * m), df = n - k)

cat("Bonferroni kvantil (alpha=0.05, m=3, df=", n-k, "):", round(krit_B, 4), "\n")
cat("(oproti bežnému qt(0.975, df=", n-k, "):", round(qt(0.975, df=n-k), 4), ")\n\n")

for (nm in names(A)) {
  a     <- A[[nm]]
  est   <- as.numeric(t(a) %*% betaHAT)
  se    <- sqrt(as.numeric(t(a) %*% covMAT %*% a))
  dolna <- est - krit_B * se
  horna <- est + krit_B * se

  cat("---", nm, "---\n")
  cat("  Odhadnutý kontrast a'beta:", round(est,   4), "\n")
  cat("  Bonferroni IS 95%:       [", round(dolna, 4), ",", round(horna, 4), "]\n")
  if (dolna > 0) {
    cat("  Záver: IS neobsahuje 0, dolná hranica > 0 — prvý faktor má štatisticky väčší vplyv\n\n")
  } else if (horna < 0) {
    cat("  Záver: IS neobsahuje 0, horná hranica < 0 — druhý faktor má štatisticky väčší vplyv\n\n")
  } else {
    cat("  Záver: IS obsahuje 0 — nedostatok dôkazov pre rozdiel medzi faktormi na hladine 5%\n\n")
  }
}

Test o heteroskedasticite 


In [26]:
#Kategoria C — Whiteov test heteroskedasticity
#Model: Happiness_Score ~ Family + Economy + Health + Freedom
epsilonHAT <- resid(model)

#Umely (artificial) model pre Whiteov test:
#obsahuje povodne regressory, ich stvorce a krizove cleny
artificial <- lm(epsilonHAT^2 ~ 
                   Family + Economy + Health + Freedom +
                   I(Family^2) + I(Economy^2) + I(Health^2) + I(Freedom^2) +
                   I(Family * Economy) + I(Family * Health) + I(Family * Freedom) +
                   I(Economy * Health) + I(Economy * Freedom) +
                   I(Health * Freedom),
                 data = df)

#pocet pozorovani
n <- nrow(df)

#pocet parametrov v artificial modeli vratane interceptu:
#intercept + 4 linearne + 4 stvorce + 6 krizove cleny = 15
cosi <- 15

#Asymptoticky Whiteov test
R2_art <- summary(artificial)$r.squared
stat <- n * R2_art
kvantil <- qchisq(0.95, df = cosi - 1)

cat("\nWHITEOV TEST HETEROSKEDASTICITY\n")
cat("H0: chyby su homoskedasticke\n")
cat("H1: chyby su heteroskedasticke\n\n")

cat("R^2 artificial modelu:", R2_art, "\n")
cat("Testovacia statistika n*R^2:", stat, "\n")
cat("Kriticka hodnota chi-square:", kvantil, "\n")

if (stat > kvantil) {
  cat("Zaver: Zamietame H0, v modeli je pritomna heteroskedasticita.\n")
} else {
  cat("Zaver: Nezamietame H0, heteroskedasticita sa nepreukazala.\n")
}

#Exaktny F-test
RSS_art <- sum(resid(artificial)^2)
RSS_art0 <- sum(resid(lm(epsilonHAT^2 ~ 1))^2)

Fprime <- ((RSS_art0 - RSS_art) / (cosi - 1)) / (RSS_art / (n - cosi))
F_kvantil <- qf(0.95, df1 = cosi - 1, df2 = n - cosi)

cat("\nEXAKTNY F-TEST\n")
cat("F' statistika:", Fprime, "\n")
cat("Kriticka hodnota F:", F_kvantil, "\n")

if (Fprime > F_kvantil) {
  cat("Zaver: Zamietame H0, v modeli je pritomna heteroskedasticita.\n")
} else {
  cat("Zaver: Nezamietame H0, heteroskedasticita sa nepreukazala.\n")
}


WHITEOV TEST HETEROSKEDASTICITY
H0: chyby su homoskedasticke
H1: chyby su heteroskedasticke

R^2 artificial modelu: 0.2324974 
Testovacia statistika n*R^2: 11.62487 
Kriticka hodnota chi-square: 23.68479 
Zaver: Nezamietame H0, heteroskedasticita sa nepreukazala.

EXAKTNY F-TEST
F' statistika: 0.7573179 
Kriticka hodnota F: 1.98581 
Zaver: Nezamietame H0, heteroskedasticita sa nepreukazala.
